# ResNet — CIFAR-10

Dikonversi dari `05_resnet_cifar10.py`. Jalankan sel secara berurutan dengan kernel Python yang memiliki `torch` dan `torchvision`. Sel terakhir memulai pelatihan.

Dataset diunduh otomatis ke `./data`, relatif terhadap direktori kerja kernel.


## Import dan konfigurasi


In [ ]:
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
BATCH_SIZE = 64
EPOCHS = 50
TRAIN_SAMPLES = 5000
VAL_SAMPLES = 1000

## Seed untuk reproduksibilitas


In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)



## Dataset dan DataLoader


In [ ]:
def get_dataloaders():
    transform_train = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                            (0.2470, 0.2435, 0.2616))
    ])

    transform_test = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                            (0.2470, 0.2435, 0.2616))
    ])

    train_set = datasets.CIFAR10("./data", train=True, download=True, transform=transform_train)
    val_set = datasets.CIFAR10("./data", train=False, download=True, transform=transform_test)

    g = torch.Generator().manual_seed(SEED)
    train_idx = torch.randperm(len(train_set), generator=g)[:TRAIN_SAMPLES]
    val_idx = torch.randperm(len(val_set), generator=g)[:VAL_SAMPLES]

    train_loader = DataLoader(
        Subset(train_set, train_idx),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        Subset(val_set, val_idx),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    return train_loader, val_loader, 10



## Fungsi pelatihan dan evaluasi


In [ ]:
def train_model(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

def run_training(model, model_name):
    set_seed()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_loader, val_loader, num_classes = get_dataloaders()

    model = model(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": []
    }

    print(f"Model: {model_name}")
    print("Dataset: CIFAR-10")
    print(f"Device: {device}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_model(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

    return model, history, val_loader, device




## Arsitektur model


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        identity = self.shortcut(x)
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return torch.relu(x + identity)

class ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.layers = nn.Sequential(
            ResidualBlock(32, 32),
            ResidualBlock(32, 64, stride=2),
            ResidualBlock(64, 64),
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128)
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layers(x)
        return self.head(x)



## Jalankan pelatihan


In [ ]:
trained_model, history, val_loader, device = run_training(ResNet, "ResNet")


In [ ]:
import matplotlib.pyplot as plt

if "history" not in globals() or not history["train_loss"]:
    raise RuntimeError("Jalankan ulang sel fungsi pelatihan dan sel pelatihan terlebih dahulu.")

epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs, [acc * 100 for acc in history["train_acc"]], label="Train")
axes[0].plot(epochs, [acc * 100 for acc in history["val_acc"]], label="Validation")
axes[0].set_title("LeNet — Accuracy")
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_ylim(0, 100)

axes[1].plot(epochs, history["train_loss"], label="Train")
axes[1].plot(epochs, history["val_loss"], label="Validation")
axes[1].set_title("LeNet — Loss")
axes[1].set_ylabel("Cross-entropy loss")

for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

@torch.no_grad()
def get_confusion_matrix(model, loader, device, num_classes):
    model.eval()
    matrix = torch.zeros((num_classes, num_classes), dtype=torch.int64)
    for images, labels in loader:
        predictions = model(images.to(device)).argmax(dim=1).cpu()
        indices = labels.cpu() * num_classes + predictions
        matrix += torch.bincount(indices, minlength=num_classes ** 2).reshape(num_classes, num_classes)
    return matrix

if any(name not in globals() for name in ("trained_model", "val_loader", "device")):
    raise RuntimeError("Jalankan ulang sel fungsi pelatihan dan sel pelatihan terlebih dahulu.")

class_names = val_loader.dataset.dataset.classes
num_classes = len(class_names)
confusion_matrix = get_confusion_matrix(trained_model, val_loader, device, num_classes)

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(confusion_matrix.numpy(), cmap="Blues", interpolation="nearest")
fig.colorbar(image, ax=ax, label="Jumlah gambar")
ax.set_xticks(range(num_classes), labels=class_names, rotation=45, ha="right")
ax.set_yticks(range(num_classes), labels=class_names)
ax.set_xlabel("Kelas prediksi")
ax.set_ylabel("Kelas aktual")
ax.set_title("LeNet — Confusion Matrix (Validation)")

threshold = confusion_matrix.max().item() / 2
for row in range(num_classes):
    for col in range(num_classes):
        count = confusion_matrix[row, col].item()
        ax.text(col, row, str(count), ha="center", va="center",
                color="white" if count > threshold else "black")

plt.tight_layout()
plt.show()
